# Type 0 (image-level) intervention: qualitative examples (Either Dog or Car)

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import torch

from utils import obj_enc_aggregation
from intervention_image_level import (
    build_laion_lexicon, load_intervention_testset, load_gt_categories, load_category_id_by_name,
    load_checkpoint, load_variants, load_boundary_type, apply_intervention,
    balanced_accuracy, positive_variant_accuracy, nb_accuracy,
)
from intervention_image_level_coco import build_trivial_lexicon

ENC = "../encodings/cocologic"
OBJ = "obj_7_crop_min_0.01_max_0.85_score_0.2_iou_0.5_model_mask-rcnn_0.25"
TRAIN_JSON = "../data/cocologic/cocologic_train.json"
TEST_JSON = "../data/cocologic/cocologic_test.json"
K = 7
AGGREGATIONS = ["sum", "max"]

VOCABS = {
    "laion": {
        "lexicon": build_laion_lexicon(),
        "base": f"{ENC}/clip/splice_embeddings_0.25.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}.h5",
        "ckpt_tag": "",
    },
    "coco_categories": {
        "lexicon": build_trivial_lexicon("../datasets/coco/annotations/instances_val2017.json"),
        "base": f"{ENC}/clip/splice_embeddings_0.25_coco_categories.h5",
        "obj": f"{ENC}/obj_encodings/clip/{OBJ}_coco_categories.h5",
        "ckpt_tag": "_coco_categories",
    },
}
for vocab, v in VOCABS.items():
    v["targets"] = {
        agg: torch.load(f"../data/cocologic/percentile_targets_full_train_{agg}_{vocab}.pt")
        for agg in AGGREGATIONS
    }

category_id_by_name = load_category_id_by_name(TRAIN_JSON)

RULE_NAMES = ["Signal and Ride", "Double Serving", "Herd Alone", "Dog or Car", "Three of a Kind",
              "Car Majority", "Empty Seat", "Single Mode Traffic", "Personal Transport", "Surf Trip"]
PERSON_RULES = ["Herd Alone", "Empty Seat", "Personal Transport", "Surf Trip"]

In [ ]:
test_ids = None
for vocab, v in VOCABS.items():
    emb, obj, img_ids, labels = load_intervention_testset(v["base"], v["obj"], K)
    v["test"] = {"emb": emb, "obj": obj[:, :K]}
    v["raw_min_test"] = min(emb.min().item(), obj.min().item())
    if test_ids is None:
        test_ids, test_labels = img_ids, labels.float()
    assert torch.equal(img_ids, test_ids)

gt_test = load_gt_categories(TEST_JSON, test_ids)
variants = load_variants(TEST_JSON, test_ids)
boundary_type = load_boundary_type(TEST_JSON, test_ids)
print(len(test_ids), "test images")

In [ ]:
def per_rule(logits):
    return {
        "B-Acc": balanced_accuracy(logits, test_labels),
        "PosVar-Acc": positive_variant_accuracy(logits, variants),
        "NB-Acc": nb_accuracy(logits, boundary_type),
    }

def rule_mean(logits):
    return {k: 100 * torch.nanmean(v).item() for k, v in per_rule(logits).items()}

def predict(model, x):
    with torch.no_grad():
        return model(x)

outputs, results = {}, []
for vocab, v in VOCABS.items():
    for agg in AGGREGATIONS:
        model = load_checkpoint(f"../checkpoints/cocologic/{agg}_super_multilabel_seed42{v['ckpt_tag']}.pt")
        base = obj_enc_aggregation(agg, v["test"]["obj"], v["test"]["emb"], only_objects=False)
        full = apply_intervention(base, gt_test, v["targets"][agg], category_id_by_name, suppress=True)
        logits = {"none": predict(model, base), "boost+suppress": predict(model, full)}
        outputs[(vocab, agg)] = {"model": model, "base": base, "full": full, "logits": logits}
        for cond, lg in logits.items():
            results.append({"vocab": vocab, "agg": agg, "intervention": cond, **rule_mean(lg)})

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image

RULE = RULE_NAMES.index("Dog or Car")
DOG, CAR = category_id_by_name["dog"], category_id_by_name["car"]
ids = test_ids.flatten().tolist()
row_of = {img_id: i for i, img_id in enumerate(ids)}

with open("../datasets/coco/annotations/instances_val2017.json") as f:
    instances = json.load(f)
image_area = {im["id"]: im["width"] * im["height"] for im in instances["images"]}
max_box_share = {}
for a in instances["annotations"]:
    if a["iscrowd"]:
        continue
    key = (a["image_id"], a["category_id"])
    share = a["bbox"][2] * a["bbox"][3] / image_area[a["image_id"]]
    max_box_share[key] = max(max_box_share.get(key, 0.0), share)
del instances
with open(TEST_JSON) as f:
    file_names = {int(k): v["file_name"] for k, v in json.load(f)["images"].items()}

def prob(vocab, agg, cond, i):
    return torch.sigmoid(outputs[(vocab, agg)]["logits"][cond][i, RULE]).item()

def status(vocab, agg, i):
    label = bool(test_labels[i, RULE])
    before = (prob(vocab, agg, "none", i) > 0.5) == label
    after = (prob(vocab, agg, "boost+suppress", i) > 0.5) == label
    return {(False, True): "fixed", (True, False): "broken", (True, True): "still correct",
            (False, False): "still wrong"}[(before, after)]

def group_value(vocab, agg, key, i, cat):
    # mean over the category's concept dimensions (one dimension for COCO)
    dims = VOCABS[vocab]["lexicon"][cat]
    return outputs[(vocab, agg)][key][i, dims].mean().item()

## Examples (COCO, sum)



In [ ]:
import h5py
import numpy as np
import torchvision.transforms as T
from torchvision.models.detection import maskrcnn_resnet50_fpn
from utils import filter_boxes

EXAMPLES = {"a": 273642, "b": 421455}
VOCAB, AGG = "coco_categories", "sum"
name_by_id = {i: n for n, i in category_id_by_name.items()}

with h5py.File(VOCABS[VOCAB]["obj"], "r") as f:
    h5_ids = f["test_obj_img_ids"][:].flatten().tolist()
    stored_boxes = {img_id: f["test_obj_boxes"][f"obj_boxes_{h5_ids.index(img_id)}"][:] for img_id in EXAMPLES.values()}


detector = maskrcnn_resnet50_fpn(pretrained=True).eval()
to_input = T.Compose([T.Resize((224, 224)), T.ToTensor()])
detections = {}
for img_id in EXAMPLES.values():
    image = to_input(Image.open(f"../datasets/coco/images/{file_names[img_id]}").convert("RGB"))
    with torch.no_grad():
        o = detector([image])[0]
    h, w = image.shape[1:]
    keep = filter_boxes(o["boxes"], o["scores"], w * h, iou_threshold=0.5, min_size=0.01, max_size=0.85,
                        minimum_score=0.5)
    keep = [k for k in keep if o["scores"][k] > 0.2][:7]
    detections[img_id] = [(name_by_id[int(o["labels"][k])], o["scores"][k].item(), o["boxes"][k]) for k in keep]

for key, img_id in EXAMPLES.items():
    i = row_of[img_id]
    gt = {name_by_id[c]: int(gt_test[i, c]) for c in torch.nonzero(gt_test[i]).flatten().tolist()}
    boxes = np.array([b.numpy() for _, _, b in detections[img_id]]).reshape(-1, 4)
    match = boxes.shape == stored_boxes[img_id].shape and np.allclose(boxes, stored_boxes[img_id], atol=1.0)
    print(f"({key}) image {img_id}, status {status(VOCAB, AGG, i)}, GT label Dog or Car = {int(test_labels[i, RULE])}")
    print("  GT categories (count):", gt)
    print("  detections (class, score):", [(c, round(s, 3)) for c, s, _ in detections[img_id]])
    print(f"  re-run boxes match stored boxes: {match} ({len(boxes)} re-run, {len(stored_boxes[img_id])} stored)")
    print(f"  P(Dog or Car) before -> after: {prob(VOCAB, AGG, 'none', i):.3f} -> "
          f"{prob(VOCAB, AGG, 'boost+suppress', i):.3f}")
    for cat in ["dog", "car"]:
        print(f"  concept {cat} before -> after: {group_value(VOCAB, AGG, 'base', i, cat):.4f} -> "
              f"{group_value(VOCAB, AGG, 'full', i, cat):.4f}")
    print(f"  largest GT box / image area: dog {100 * max_box_share.get((img_id, DOG), 0.0):.1f} %, "
          f"car {100 * max_box_share.get((img_id, CAR), 0.0):.1f} %")
    print()

In [ ]:
import matplotlib.patches as patches

# GT boxes of the two example images 
with open("../datasets/coco/annotations/instances_val2017.json") as f:
    example_anns = [a for a in json.load(f)["annotations"]
                    if a["image_id"] in EXAMPLES.values() and not a["iscrowd"]]

BOX_COLORS_V2 = {"dog": "#2a78d6", "car": "#eb6834"}
OTHER_COLOR_V2 = "#1baf7a"  # all other GT categories
TEXTWIDTH_IN = 6.3
plt.rcParams.update({"font.size": 10})

images = {img_id: Image.open(f"../datasets/coco/images/{file_names[img_id]}").convert("RGB")
          for img_id in EXAMPLES.values()}
aspects = [images[img_id].width / images[img_id].height for img_id in EXAMPLES.values()]

fig_v2 = plt.figure(figsize=(TEXTWIDTH_IN, 4.3), facecolor="white")
grid = fig_v2.add_gridspec(2, 2, width_ratios=aspects, height_ratios=[1, 0.42], wspace=0.06, hspace=0.04)
example_tables = {}
MARK_COLORS = {True: "#0ca30c", False: "#d03b3b"}
marks = []
for col, (key, img_id) in enumerate(EXAMPLES.items()):
    i = row_of[img_id]
    ax = fig_v2.add_subplot(grid[0, col])
    ax.imshow(images[img_id])

    img_w, img_h = images[img_id].size
    pos = ax.get_position()
    px_per_inch = max(img_w / (pos.width * TEXTWIDTH_IN), img_h / (pos.height * fig_v2.get_figheight()))
    margin = 1.5 / 72 * px_per_inch
    for a in example_anns:
        if a["image_id"] != img_id:
            continue
        name = name_by_id[a["category_id"]]
        x, y, w, h = a["bbox"]
        x0, y0 = max(x, margin), max(y, margin)
        x1, y1 = min(x + w, img_w - margin), min(y + h, img_h - margin)
        if (x0, y0, x1, y1) != (x, y, x + w, y + h):
            print(f"({key}) {name} box moved inwards by up to {max(x0 - x, y0 - y, x + w - x1, y + h - y1):.1f} px")
        x, y, w, h = x0, y0, x1 - x0, y1 - y0
        color = BOX_COLORS_V2.get(name, OTHER_COLOR_V2)
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=3))
        ax.text(x, y, name, color="white", fontsize=9, va="bottom",
                bbox=dict(facecolor=color, edgecolor="none", pad=1))
    ax.set_axis_off()
    ax.annotate(f"({key})", xy=(0, 1), xycoords="axes fraction", xytext=(0, 15), textcoords="offset points",
                fontsize=11, ha="left", va="bottom")

    values = {
        "dog": (group_value(VOCAB, AGG, "base", i, "dog"), group_value(VOCAB, AGG, "full", i, "dog")),
        "car": (group_value(VOCAB, AGG, "base", i, "car"), group_value(VOCAB, AGG, "full", i, "car")),
        "P(rule)": (prob(VOCAB, AGG, "none", i), prob(VOCAB, AGG, "boost+suppress", i)),
    }
    example_tables[(key, img_id)] = values
    tax = fig_v2.add_subplot(grid[1, col])

    col_width = 0.6 / (tax.get_position().width * TEXTWIDTH_IN)
    tax.set_axis_off()
    cell_text = [[name, f"{b:.3f}", f"{a:.3f}"] for name, (b, a) in values.items()]
    table = tax.table(cellText=cell_text, colLabels=["", "Before", "After"],
                      loc="upper center", cellLoc="center", colWidths=[col_width] * 3)
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    for cell in table.get_celld().values():
        cell.set_edgecolor("black")
        cell.set_linewidth(0.6)
        cell.set_facecolor("white")
    table[0, 0].set_visible(False)
    for row, name in enumerate(values, start=1):
        table[row, 0].get_text().set_color(BOX_COLORS_V2.get(name, "black"))
        table[row, 0].get_text().set_horizontalalignment("left")

    correct = (values["P(rule)"][1] > 0.5) == bool(test_labels[i, RULE])
    marks.append((table[3, 2], "\u2713" if correct else "\u2717", MARK_COLORS[correct]))

fig_v2.canvas.draw()
renderer = fig_v2.canvas.get_renderer()
for cell, mark, color in marks:
    bb = cell.get_window_extent(renderer).transformed(fig_v2.transFigure.inverted())
    value = cell.get_text().get_text()
    cell.get_text().set_text("")
    y = (bb.y0 + bb.y1) / 2
    fig_v2.text(bb.x0 + 0.42 * bb.width, y, value, fontsize=9, ha="center", va="center")
    fig_v2.text(bb.x0 + 0.84 * bb.width, y, mark, color=color, fontsize=14, fontweight="bold", fontfamily="DejaVu Sans",
                ha="center", va="center")

for (key, img_id), values in example_tables.items():
    print(f"({key}) {img_id}: " + ", ".join(f"{k} {b:.4f} -> {a:.4f}" for k, (b, a) in values.items()))
plt.show()

In [ ]:
import os

os.makedirs("../figures", exist_ok=True)
fig_v2.savefig("../figures/type0_examples.pdf", bbox_inches="tight")
print("saved ../figures/type0_examples.pdf")